# 📊 Exploratory Data Analysis — Obesity Risk Estimation

**Dataset:** [UCI — Estimation of Obesity Levels Based on Eating Habits & Physical Condition](https://archive.ics.uci.edu/dataset/544)

**Authors:** GR 27 — École Centrale Casablanca

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_processing import fetch_dataset, optimize_memory

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
%matplotlib inline

## 1 · Dataset Overview

In [ ]:
df = fetch_dataset()
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 2 · Missing Value Detection

In [ ]:
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values ✅')

The dataset has **no missing values**, which is expected since 77 % of it was synthetically generated.

## 3 · Outlier Detection

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

fig, axes = plt.subplots(2, (len(numeric_cols)+1)//2, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.boxplot(y=df[col], ax=axes[i], color='#42a5f5')
    axes[i].set_title(col)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Box Plots — Outlier Detection', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

Some features show mild outliers (e.g. *Weight*, *Age*), but since they represent plausible real-world values we keep them.

## 4 · Class Distribution Analysis (7 Obesity Levels)

In [ ]:
target = 'NObeyesdad'
class_counts = df[target].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors = sns.color_palette('Blues_d', n_colors=len(class_counts))
class_counts.plot.bar(ax=axes[0], color=colors, edgecolor='white')
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=35)

class_counts.plot.pie(ax=axes[1], autopct='%1.1f%%', colors=colors, startangle=140)
axes[1].set_ylabel('')
axes[1].set_title('Class Proportions')

plt.tight_layout()
plt.show()

print(class_counts)

### Class Imbalance Analysis

The classes are **relatively balanced** thanks to the SMOTE augmentation applied during dataset creation.  
The smallest class has ~270 samples and the largest ~350, so the ratio is mild (~1.3:1).  
We still use `class_weight='balanced'` in our classifiers as a precaution.

## 5 · Correlation Analysis

In [ ]:
# Encode categoricals temporarily for correlation
from sklearn.preprocessing import LabelEncoder

df_enc = df.copy()
for col in df_enc.select_dtypes(include='object').columns:
    df_enc[col] = LabelEncoder().fit_transform(df_enc[col])

plt.figure(figsize=(14, 10))
corr = df_enc.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, square=True)
plt.title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

**Key observations:**
- *Weight* has the strongest positive correlation with the obesity target.
- *Height*, *family history*, and *FCVC* also show notable correlations.
- No extremely high inter-feature correlations → multicollinearity is not a concern.

## 6 · Feature Distributions by Obesity Level

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
key_feats = ['Age', 'Weight', 'Height', 'FCVC', 'FAF', 'CH2O']

for ax, feat in zip(axes.flatten(), key_feats):
    for level in df[target].unique():
        subset = df[df[target] == level]
        ax.hist(subset[feat], bins=20, alpha=0.5, label=level)
    ax.set_title(feat)
    ax.legend(fontsize=7, loc='upper right')

plt.suptitle('Feature Distributions by Obesity Level', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 7 · Memory Optimization Demonstration

In [ ]:
print('Before optimization:')
print(df.dtypes)
print(f'\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.4f} MB\n')

df_opt = optimize_memory(df)

print('\nAfter optimization:')
print(df_opt.dtypes)

## 8 · Key Take-aways

| Aspect | Finding |
|---|---|
| **Missing values** | None |
| **Outliers** | Mild, plausible — no removal needed |
| **Class balance** | Approximately balanced (SMOTE applied in original data) |
| **Top correlated feature** | *Weight* → strongest predictor |
| **Memory optimisation** | ~40-50 % reduction via dtype downcasting |